# Practice Lab: Math Foundations 1 - The Basics

Lesson 1.1 of the Netsetos GenAI Engineering course. 8 exercises across vectors, cosine similarity, softmax, and temperature. Run each cell. Fill in the `...` and check against the expected output.

In [ ]:
# Setup - all exercises are local NumPy. No API keys required.
!pip install -q numpy scikit-learn matplotlib

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
print('numpy:', np.__version__)

## Exercise 1 - Vector arithmetic warm-up (Easy)

Compute `paris - france + italy`. Identify the closest known capital.

In [ ]:
paris  = np.array([0.7,  0.5])
france = np.array([0.6,  0.4])
italy  = np.array([0.6, -0.3])
rome   = np.array([0.7, -0.2])
berlin = np.array([0.5,  0.8])

# YOUR CODE HERE
# result = paris - france + italy
# print result, then the distance to rome and to berlin (np.linalg.norm)
result = ...

# Expected:
# result: [0.7 -0.2]
# dist to rome:   0.0
# dist to berlin: 1.04

<details><summary>💡 Hint</summary>

`np.linalg.norm(x - y)` gives the Euclidean distance between two vectors.
</details>

**✅ Criteria:** result printed, distances to rome and berlin printed, rome is the closest capital.

<details><summary>✅ Solution</summary>

In [ ]:
paris  = np.array([0.7,  0.5])
france = np.array([0.6,  0.4])
italy  = np.array([0.6, -0.3])
rome   = np.array([0.7, -0.2])
berlin = np.array([0.5,  0.8])

result = paris - france + italy
print('result:', result)
print('dist to rome:  ', np.linalg.norm(result - rome))
print('dist to berlin:', np.linalg.norm(result - berlin))
# Expected:
# result: [0.7 -0.2]
# dist to rome:   0.0
# dist to berlin: 1.04

</details>

---

## Exercise 2 - Hand-coded cosine vs sklearn (Easy)

Implement cosine from scratch with epsilon guard. Verify against sklearn.

In [ ]:
def cosine(a, b, eps=1e-6):
    # YOUR CODE HERE
    # dot product divided by the product of L2 norms, with eps in the denominator
    ...

np.random.seed(0)
a, b = np.random.randn(5), np.random.randn(5)
# compute your cosine and sklearn's cosine_similarity, print both, assert they match within 1e-5

<details><summary>💡 Hint</summary>

`np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps)`. sklearn: `cosine_similarity([a], [b])[0][0]`.
</details>

**✅ Criteria:** cosine implemented with the eps guard; hand-coded and sklearn values printed and match within 1e-5.

<details><summary>✅ Solution</summary>

In [ ]:
def cosine(a, b, eps=1e-6):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps)

np.random.seed(0)
a, b = np.random.randn(5), np.random.randn(5)
mine = cosine(a, b)
sk   = cosine_similarity([a], [b])[0][0]
print(f'hand-coded: {mine:.6f}')
print(f'sklearn:    {sk:.6f}')
assert abs(mine - sk) < 1e-5

</details>

---

## Exercise 3 - Top-K retrieval (Easy)

10 random doc vectors, one query, top-3 by cosine.

In [ ]:
np.random.seed(42)
docs  = np.random.randn(10, 8)
query = np.random.randn(8)

# YOUR CODE HERE
# score every doc against the query with cosine (reuse the function from Exercise 2),
# then print the top-3 doc ids and scores, highest first

<details><summary>💡 Hint</summary>

Build a list of `(doc_id, cosine(query, docs[i]))`, then `sorted(scores, key=lambda kv: kv[1], reverse=True)[:3]`.
</details>

**✅ Criteria:** all 10 docs scored; top-3 printed highest-first with cosine scores.

<details><summary>✅ Solution</summary>

In [ ]:
np.random.seed(42)
docs  = np.random.randn(10, 8)
query = np.random.randn(8)

scores = [(i, cosine(query, docs[i])) for i in range(len(docs))]
top3 = sorted(scores, key=lambda kv: kv[1], reverse=True)[:3]
for doc_id, score in top3:
    print(f'doc {doc_id}: cos={score:.3f}')

</details>

---

## Exercise 4 - Softmax stability test (Medium)

The NaN demo. Compare naive softmax (overflow) to stable softmax (max-shift).

In [ ]:
def naive_softmax(x):
    # YOUR CODE HERE - the textbook formula, no max-shift
    ...

def stable_softmax(x):
    # YOUR CODE HERE - subtract np.max(x) before the exp
    ...

logits = np.array([1000.0, 1001.0, 1002.0])
import warnings; warnings.filterwarnings('ignore')
# print naive vs stable softmax on these logits
# Expected:
# naive:  [nan nan nan]
# stable: [0.090 0.245 0.665]

<details><summary>💡 Hint</summary>

Naive: `e = np.exp(x); e / e.sum()`. Stable: `e = np.exp(x - np.max(x)); e / e.sum()`.
</details>

**✅ Criteria:** both variants implemented; naive overflows to nan; stable returns [0.090 0.245 0.665].

<details><summary>✅ Solution</summary>

In [ ]:
def naive_softmax(x):
    e = np.exp(x)
    return e / e.sum()

def stable_softmax(x):
    shifted = x - np.max(x)
    e = np.exp(shifted)
    return e / e.sum()

logits = np.array([1000.0, 1001.0, 1002.0])
import warnings; warnings.filterwarnings('ignore')
print('naive: ', naive_softmax(logits))
print('stable:', stable_softmax(logits))
# Expected:
# naive:  [nan nan nan]
# stable: [0.090 0.245 0.665]

</details>

---

## Exercise 5 - Temperature sweep + entropy (Medium)

For logits [3.0, 2.5, 1.0, 0.5], print softmax + entropy at T = 0.1 / 0.5 / 1.0 / 1.5 / 2.0. Entropy is monotonically increasing.

In [ ]:
def softmax_T(x, T=1.0):
    # YOUR CODE HERE - divide the logits by T, then run a stable softmax
    ...

def entropy(p):
    # YOUR CODE HERE - -sum(p * log(p)); clip p away from 0 first
    ...

logits = np.array([3.0, 2.5, 1.0, 0.5])
# for T in [0.1, 0.5, 1.0, 1.5, 2.0]: print the distribution and its entropy

<details><summary>💡 Hint</summary>

`softmax_T`: `s = x / T; s -= s.max(); e = np.exp(s); return e / e.sum()`. Entropy climbs as T rises.
</details>

**✅ Criteria:** softmax_T and entropy implemented; five temperatures swept; entropy increases with T.

<details><summary>✅ Solution</summary>

In [ ]:
def softmax_T(x, T=1.0):
    s = x / T
    s = s - np.max(s)
    e = np.exp(s)
    return e / e.sum()

def entropy(p):
    p = np.clip(p, 1e-12, 1.0)
    return float(-np.sum(p * np.log(p)))

logits = np.array([3.0, 2.5, 1.0, 0.5])
for T in [0.1, 0.5, 1.0, 1.5, 2.0]:
    p = softmax_T(logits, T)
    print(f'T={T:.1f}  p={[f"{x:.3f}" for x in p]}  H={entropy(p):.3f}')

</details>

---

## Exercise 6 - Cosine vs Euclidean overlap (Medium)

20 random vectors, query, top-5 by each metric, count overlap.

In [ ]:
np.random.seed(7)
vecs = np.random.randn(20, 8)
query = np.random.randn(8)

# YOUR CODE HERE
# rank the 20 vectors by cosine (higher = closer) and by euclidean (smaller distance = closer),
# take the top-5 ids from each, then count how many ids appear in both

<details><summary>💡 Hint</summary>

For euclidean rank by `-np.linalg.norm(query - vecs[i])` so larger is closer. Overlap: `set(top5_cos) & set(top5_euc)`.
</details>

**✅ Criteria:** top-5 by cosine, top-5 by euclidean, and the overlap count all printed.

<details><summary>✅ Solution</summary>

In [ ]:
np.random.seed(7)
vecs = np.random.randn(20, 8)
query = np.random.randn(8)

cos_scores = [(i, cosine(query, vecs[i])) for i in range(20)]
euc_scores = [(i, -np.linalg.norm(query - vecs[i])) for i in range(20)]
top5_cos = [i for i,_ in sorted(cos_scores, key=lambda kv: kv[1], reverse=True)[:5]]
top5_euc = [i for i,_ in sorted(euc_scores, key=lambda kv: kv[1], reverse=True)[:5]]
overlap = set(top5_cos) & set(top5_euc)
print('by cosine:   ', top5_cos)
print('by euclidean:', top5_euc)
print('overlap count:', len(overlap))

</details>

---

## Exercise 7 - Sampling histogram convergence (Hard)

Draw 10,000 samples from softmax([2.0, 1.0, 0.1]). Empirical should match theoretical to within 1%.

In [ ]:
logits = np.array([2.0, 1.0, 0.1])
theoretical = softmax_T(logits, 1.0)

np.random.seed(0)
N = 10000
# YOUR CODE HERE
# draw N samples from the categorical distribution `theoretical` with np.random.choice,
# build the empirical frequencies, then print theoretical vs empirical and the max abs difference

<details><summary>💡 Hint</summary>

`np.random.choice(3, size=N, p=theoretical)`, then `np.bincount(samples, minlength=3) / N`.
</details>

**✅ Criteria:** 10,000 samples drawn; empirical matches theoretical within 1%.

<details><summary>✅ Solution</summary>

In [ ]:
logits = np.array([2.0, 1.0, 0.1])
theoretical = softmax_T(logits, 1.0)

np.random.seed(0)
N = 10000
samples = np.random.choice(3, size=N, p=theoretical)
empirical = np.bincount(samples, minlength=3) / N

print('theoretical:', [f'{p:.3f}' for p in theoretical])
print('empirical:  ', [f'{p:.3f}' for p in empirical])
print('max diff:   ', float(np.max(np.abs(theoretical - empirical))))
assert np.max(np.abs(theoretical - empirical)) < 0.01

</details>

---

## Exercise 8 - Greedy vs sampled n-grams (Hard)

The Wasowski finding in microcosm. Greedy produces 1-2 distinct bigrams. Sampling produces 10+.

In [ ]:
VOCAB = ['the', 'cat', 'sat', 'on', 'mat', 'and', 'a', 'dog', 'ran']

def get_logits(history):
    base = np.array([3.0, 1.5, 1.0, 1.0, 0.8, 0.5, 0.5, 0.4, 0.3])
    if history and history[-1] == 'the':
        base[1] += 1.5
    return base

def generate(T, n_tokens=50, seed=0):
    np.random.seed(seed)
    history = []
    for _ in range(n_tokens):
        # YOUR CODE HERE
        # T <= 0 means greedy (argmax); T > 0 means sample from softmax_T(logits, T)
        ...
    return history

def distinct_bigrams(tokens):
    # YOUR CODE HERE - count the unique (tokens[i], tokens[i+1]) pairs
    ...

# generate greedy (T=0) and sampled (T=1.0), then print the distinct bigram counts and first 10 greedy tokens

<details><summary>💡 Hint</summary>

Greedy: `int(p.argmax())`. Sampled: `np.random.choice(len(p), p=p)`. Bigrams: `len({(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)})`.
</details>

**✅ Criteria:** generate handles both greedy and sampled; distinct_bigrams implemented; greedy yields ~1-2 bigrams vs 10+ when sampled.

<details><summary>✅ Solution</summary>

In [ ]:
VOCAB = ['the', 'cat', 'sat', 'on', 'mat', 'and', 'a', 'dog', 'ran']

def get_logits(history):
    base = np.array([3.0, 1.5, 1.0, 1.0, 0.8, 0.5, 0.5, 0.4, 0.3])
    if history and history[-1] == 'the':
        base[1] += 1.5
    return base

def generate(T, n_tokens=50, seed=0):
    np.random.seed(seed)
    history = []
    for _ in range(n_tokens):
        p = softmax_T(get_logits(history), T if T > 0 else 0.001)
        idx = int(p.argmax()) if T <= 0 else np.random.choice(len(p), p=p)
        history.append(VOCAB[idx])
    return history

def distinct_bigrams(tokens):
    return len({(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)})

greedy   = generate(T=0,   n_tokens=50)
sampled  = generate(T=1.0, n_tokens=50)
print('greedy bigrams:  ', distinct_bigrams(greedy), '/49')
print('sampled bigrams: ', distinct_bigrams(sampled), '/49')
print('first 10 greedy: ', greedy[:10])

</details>

---

## Wrap-up

Four math primitives, eight exercises. Exercise 4 (NaN softmax) and Exercise 8 (greedy repetition) are the two production bugs you are most likely to encounter. The next lesson (1.2) takes the softmax built here and uses it inside cross-entropy loss for gradient descent.